In [9]:
import os

import asyncio
import dotenv
from rich import print as rich_print
from mlflow_tracing import enable_mlflow_tracing

from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import ToolRetryMiddleware

from deepagents.middleware.filesystem import FilesystemMiddleware

dotenv.load_dotenv()
# By default, run_tracer_inline is enabled and known benign warning noise is filtered.
enable_mlflow_tracing(run_tracer_inline=True)
grafana_mcp_endpoint = os.getenv('GRAFANA_MCP_ENDPOINT')

with open("./prompt/agent-instruction.md", "r", encoding="utf-8") as f:
    AGENT_SYSTEM_PROMPT = f.read()
    
MODEL = 'gemini-3.1-pro-preview'
MODEL_PROVIDER = 'google_genai'
MCP_CONFIG = {
    'grafana': {
        'url': grafana_mcp_endpoint,
        'transport': 'http',
    }
}

In [10]:
llm = init_chat_model(model=MODEL, 
                      model_provider=MODEL_PROVIDER, 
                      thinking_level="high",
                      include_thoughts=True)
mcp_client = MultiServerMCPClient(MCP_CONFIG)
tools = await mcp_client.get_tools()
tool_error_middleware = ToolRetryMiddleware(max_retries=0, on_failure="continue")

print(f'Llm profile: \n{MODEL_PROVIDER}:{MODEL}\n {llm.profile}')

agent = create_agent(
    llm,
    tools,
    middleware=[tool_error_middleware],
    system_prompt=AGENT_SYSTEM_PROMPT,
)

Llm profile: 
google_genai:gemini-3.1-pro-preview
 {'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}


In [14]:
stream = agent.astream(
    {"messages": [HumanMessage(content="On March 6, 2021, between 20:30 and 21:00, a failure was observed in the system. However, the specific component responsible for this failure and the underlying reason remain unknown. You need to determine the root cause component and root cause reason for the observed failure. ")]},
    stream_mode="updates"  # 或 "updates" / "messages"
)

async for chunk in stream:
    if "messages" not in chunk:
        continue

    last_message = chunk["messages"][-1]

    # AI 回覆
    if isinstance(last_message, AIMessage):
        print("🤖 AI:", last_message.content)

        # 工具呼叫
        if last_message.tool_calls:
            print("🔧 Tool Calls:",
                  [tc["name"] for tc in last_message.tool_calls])

    # 工具結果
    elif isinstance(last_message, ToolMessage):
        print("✅ Tool Result:", last_message.name)
        print(last_message.content)